## 외부데이터 EDA

In [ ]:
from pathlib import Path

from src.utils import load_config

# notebooks에서 돌려도 최상위를 찾게
ROOT = Path.cwd()
if not (ROOT / "configs" / "base.yaml").exists():
    ROOT = ROOT.parent  # notebooks -> YacssoK

print("ROOT:", ROOT)

cfg = load_config(str(ROOT / "configs" / "base.yaml"))
d = cfg["data"]

BASE_ANN = ROOT / d["train_annotations"]
BASE_IMG_DIR = ROOT / d["train_images"]

EXT_LABEL_ROOTS = [
    ROOT / "data/aihub/labels/TL_1_조합",
    # ROOT / "data/aihub/labels/TL_3_조합",
]
EXT_IMAGE_ROOTS = [
    ROOT / "data/aihub/images/TS_1_조합",
    # ROOT / "data/aihub/images/TS_3_조합",
]

EXT_LABEL_ROOTS = [p for p in EXT_LABEL_ROOTS if p.exists()]
EXT_IMAGE_ROOTS = [p for p in EXT_IMAGE_ROOTS if p.exists()]

print("base:", BASE_ANN, BASE_ANN.exists())
print("external label roots:", EXT_LABEL_ROOTS)
print("external image roots:", EXT_IMAGE_ROOTS)

## 1. 데이터 읽기

In [ ]:
from src.dataset import build_class_map, load_records
from src.eda import dataset_summary, records_to_frames

records, categories = load_records(str(BASE_ANN))
class_map = build_class_map(records, categories)
images_df, boxes_df = records_to_frames(records, class_map)
dataset_summary(images_df, boxes_df)
print(f"base 클래스 수: {len(class_map)}")

In [ ]:
import json
import re


def num(value):
    digits = re.sub(r"\D", "", str(value or ""))
    return int(digits) if digits else None


def load_json(path):
    for enc in ("utf-8-sig", "utf-8", "cp949"):
        try:
            return json.loads(path.read_text(encoding=enc))
        except Exception:
            continue
    return None


ext_jsons = []
for root in EXT_LABEL_ROOTS:
    ext_jsons += list(root.rglob("*.json")) + list(root.rglob("*.JSON"))

print(f"ext_jsons: {len(ext_jsons)}개")

## 2. base 데이터 vs external 데이터 

2-1 외부데이터 json 스키마 확인 - 키가 기본데이터와 같은가?

In [ ]:
from pathlib import Path

for base in [ROOT / "data/aihub/labels", ROOT / "data/aihub/images"]:
    print(f"\n=== {base.relative_to(ROOT)} exists={base.exists()} ===")
    if not base.exists():
        continue
    for sub in sorted([p for p in base.iterdir()]):
        if sub.is_dir():
            n = sum(1 for _ in sub.rglob("*") if _.is_file())
            # 하위 한 겹까지만 보여주기
            inner = [p.name for p in sub.iterdir()][:5]
            print(f"- {sub.name}/ : 파일 {n}개, 안쪽 예시 {inner}")

In [ ]:
# 외부데이터 - 기본데이터 annotations 비교

# 외부 1개만
ext_file = next(p for root in EXT_LABEL_ROOTS for p in root.rglob("*.json"))
print("외부 샘플:", ext_file.relative_to(ROOT))

with open(ext_file, encoding="utf-8-sig") as f:
    ext = json.load(f)

print("\n[BASE annotations 항목]")
print(list(records[0].keys()))
print(records[0])

print("\n[EXTERNAL top]")
print(list(ext.keys()) if isinstance(ext, dict) else f"list len={len(ext)}")

if isinstance(ext, dict) and "annotations" in ext:
    print("\n[EXTERNAL annotations[0] 항목]")
    print(list(ext["annotations"][0].keys()))
    print(ext["annotations"][0])

In [ ]:
print("images[0]:", ext["images"][0] if isinstance(ext["images"], list) else ext["images"])
print("categories[:3]:", ext["categories"][:3])

인사이트: 
- base 어노테이션은 내부 변환 포맷(boxes=xyxy category_ids=진짜 약 번호)이고, 외부는 날것(bbox=xywh, category_id=1 더미)이라 그대로 합칠 수 없음. images /    annotations 구조는 동일하나 내용 스키마가 다름.
- 외부 라벨 분석 결과 area, iscrowd, segmentation, ignore는 학습에 사용하지 않는 항목으로 확인되어, bbox와 약 번호만 남기고 버리는 최소 정제를 적용할 것
- 외부 라벨에 drug_N이 확인되어, 기존 category_id는 더미값으로 무시하고 drug_N 기준으로 기본 데이터와 매칭하는 가공을 적용할 것

2-2 파일명 규칙 및 각도 분포 확인 - 70,75,90 이외의 각도가 있는가?

In [ ]:
import pandas as pd


def parse_angle(fname: str):
    # K-001001_0_2_0_2_70_000_208.png -> 6번째 칸 = 70
    try:
        return fname.split("_")[5]
    except Exception:
        return None


base_angles = pd.Series([parse_angle(r["file_name"]) for r in records]).value_counts()
print("=== base 각도 ===")
display(base_angles)

# 외부는 파일명만 긁어서 비교 (JSON 안 까도 됨)
ext_names = []
for root in EXT_LABEL_ROOTS:
    for p in root.rglob("*.json"):
        ext_names.append(p.stem)  # 이미지 파일명과 대응되는 경우가 많음
# 이미지 파일명으로 다시 확인
ext_img_names = []
for root in EXT_IMAGE_ROOTS:
    if root.exists():
        ext_img_names += [p.name for p in root.rglob("*.png")]

ext_angles = pd.Series([parse_angle(n) for n in ext_img_names]).value_counts()
print("=== 외부 각도 (이미지 파일명 기준) ===")
display(ext_angles)

인사이트: 
base도 70/75/90 세 각도만 있고, 외부도 70/75/90 세 각도만 있음 → 촬영 조건 도메인이 맞음
외부가 각도별로 1003장씩 균등이라 각도 편향 없음

2-3. 이미지 크기 + 박스 확인  - base와 같은 사진인가

In [ ]:
from PIL import Image

# base 크기 분포 (images_df에 width/height 있으면 그대로)
print("=== base 크기 ===")
display(images_df[["width", "height"]].value_counts().head(10))

# 외부 샘플 20장만 크기 확인 (전수 읽으면 느림)
sizes = []
for root in EXT_IMAGE_ROOTS:
    if not root.exists():
        continue
    for p in list(root.rglob("*.png"))[:20]:
        try:
            with Image.open(p) as im:
                sizes.append(im.size)  # (w, h)
        except Exception:
            pass

print("=== 외부 샘플 크기 (w, h) 20장 ===")
print(pd.Series(sizes).value_counts())

인사이트:
외부 샘플 20장(TS_1 20)이 전부 976×1280이고, base 232장도 전부 같은 크기라 리사이즈 없이 합칠 수 있다.

2-4 박스 비교

In [ ]:
from src.eda import bbox_stats

rows = []
bad_fmt = 0
for p in ext_jsons:
    data = load_json(p)
    if not data or not data.get("images"):
        continue
    im = data["images"][0]
    W, H = im.get("width"), im.get("height")
    if not W or not H:
        continue
    dnum = num(im.get("drug_N"))
    for a in data.get("annotations", []):
        b = a.get("bbox")
        if not isinstance(b, list) or len(b) != 4:
            bad_fmt += 1
            continue
        try:
            x, y, w, h = [float(v) for v in b]
        except Exception:
            bad_fmt += 1
            continue
        if w <= 0 or h <= 0:
            bad_fmt += 1
            continue
        rows.append(
            {
                "file_name": im.get("file_name"),
                "category_id": dnum,
                "box_w": w,
                "box_h": h,
                "area": w * h,
                "rel_area": (w * h) / (W * H),
                "aspect": w / h,
                "cx": (x + w / 2) / W,
                "cy": (y + h / 2) / H,
            }
        )

boxes_ext_full = pd.DataFrame(rows)
print(f"박스 {len(boxes_ext_full)}개, 형식 오류 {bad_fmt}개")

display(bbox_stats(boxes_df))
display(bbox_stats(boxes_ext_full))

인사이트:
확인 후 총 6011개의 bbox 중 20개 오류 검출



2-5. 클래스 확인 - base와 클래스가 겹치는가

In [ ]:
import json

# def num(value):
#     digits = re.sub(r"\D", "", str(value or ""))
#     return int(digits) if digits else None


# def load_json(path):
#     for enc in ("utf-8-sig", "utf-8", "cp949"):
#         try:
#             return json.loads(path.read_text(encoding=enc))
#         except Exception:
#             continue
#     return None


if isinstance(categories, dict):
    base_ids = {int(k) for k in categories}
elif isinstance(categories, list) and categories and isinstance(categories[0], dict):
    base_ids = {int(c["id"]) for c in categories}
else:
    base_ids = {int(c["category_id"]) for c in class_map}

sample = load_json(ext_jsons[0])
print("images[0] 키:", list(sample["images"][0]))
print("images[0]:", sample["images"][0])

drug_ids, dl_ids, failed = set(), set(), 0
for path in ext_jsons:
    data = load_json(path)
    if not data:
        failed += 1
        continue
    for image in data.get("images", []):
        drug = num(image.get("drug_N"))
        dl = num(image.get("dl_idx"))
        if drug is not None:
            drug_ids.add(drug)
        if dl is not None:
            dl_ids.add(dl)

print(f"base {len(base_ids)}종, JSON {len(ext_jsons)}개, 실패 {failed}")
print(f"drug_N {len(drug_ids)}종, known {len(drug_ids & base_ids)}, new {len(drug_ids - base_ids)}")
print(f"dl_idx {len(dl_ids)}종, known {len(dl_ids & base_ids)}, new {len(dl_ids - base_ids)}")
print("drug_N new 예시:", sorted(drug_ids - base_ids)[:15])

인사이트:  
- TL_1 라벨 6011개 전수 확인 결과 drug_N 41종 중 기본 데이터와 겹치는 번호 21종, 새로운 번호 20종으로 나뉘어, 겹치는 것만 더하는 버전과 전체를 더하는 버전 두 가지로 나눠 가공할 것
- dl_idx는 기본 데이터와 겹침이 0종이라 매칭 기준으로 쓸 수 없어, drug_N을 기준으로 기본 데이터와 매칭하는 최소 정제를 적용할 것

2-6. 기본 이미지 train/test와 겹침

In [ ]:
base_names = set(r["file_name"] for r in records)

# test 목록
from src.dataset import list_images

test_names = set(p.name for p in list_images(str(ROOT / d["test_images"])))

ext_names = set()
for p in ext_jsons:
    data = load_json(p)
    if data and data.get("images"):
        ext_names.add(data["images"][0]["file_name"])

print(f"외부 이미지: {len(ext_names)}장")
print(f"대회 train과 이름 같음: {len(ext_names & base_names)}장")
print(f"테스트와 이름 같음: {len(ext_names & test_names)}장")

## 3. 클래스 균형

In [ ]:
import pandas as pd

from src.eda import class_distribution
from src.visualize import plot_class_distribution

# def num(v):
#     d = re.sub(r"\D", "", str(v or ""))
#     return int(d) if d else None


# def load_json(path):
#     import json

#     for enc in ("utf-8-sig", "utf-8", "cp949"):
#         try:
#             return json.loads(path.read_text(encoding=enc))
#         except Exception:
#             continue
#     return None


rows = []
for p in ext_jsons:
    data = load_json(p)
    if not data:
        continue
    # 이미지 id -> drug_N, dl_name, file_name
    img_map = {}
    for im in data.get("images", []):
        dnum = num(im.get("drug_N"))
        if dnum is not None:
            img_map[im.get("id")] = (dnum, im.get("dl_name") or str(dnum), im.get("file_name"))
    for a in data.get("annotations", []):
        info = img_map.get(a.get("image_id"))
        if info:
            dnum, name, fname = info
            rows.append({"category_id": dnum, "name": name, "file_name": fname})

boxes_ext = pd.DataFrame(rows)
print(f"박스 {len(boxes_ext)}개, drug_N {boxes_ext['category_id'].nunique()}종")

class_df_ext = class_distribution(boxes_ext)
plot_class_distribution(class_df_ext)
class_df_ext.head(10)

In [ ]:
# known / new 플래그 붙이기

class_df_ext["known"] = class_df_ext["category_id"].isin(base_ids)

print("known:", class_df_ext["known"].sum(), "종")
print("new:", (~class_df_ext["known"]).sum(), "종")
display(
    class_df_ext.sort_values("share_%", ascending=False).head(10)
)  # share_% = n_boxes / 전체 n_boxes 합 * 100

3-1. 기본데이터+ 외부데이터 클래스 불균형

In [ ]:
import pandas as pd

from src.eda import class_distribution
from src.visualize import plot_class_distribution

combined = pd.concat(
    [
        boxes_df[["category_id", "name", "file_name"]],
        boxes_ext[["category_id", "name", "file_name"]],
    ],
    ignore_index=True,
)

combined_df = class_distribution(combined)
plot_class_distribution(combined_df)
combined_df.head(10)

In [ ]:
combined_df.tail(10)  # 가장 적은 클래스

인사이트:
기본데이터보다 클래스 불균형이 조금 완화되었으나, 그대로 학습에 쓸 정도는 아님.
클래스 불균형 완화 실험 D1,D2로 잡을 것